# Cell 1 - Markdown
"""
# 06 Prediction Demo

## Student Performance Prediction Using Machine Learning

This notebook demonstrates how the final trained model can be used in practice.

Main tasks:
- Load the final saved model
- Load the final threshold
- Create sample student cases
- Predict Pass/Fail
- Estimate risk level
- Provide simple intervention recommendations
"""

In [1]:
# Cell 2 - Import Libraries

import pandas as pd
import numpy as np

from pathlib import Path
import joblib
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [2]:
# Cell 3 - Define Paths

PROJECT_ROOT = Path("..")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "StudentPerformanceFactors_cleaned.csv"

MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

FINAL_MODEL_PATH = MODELS_DIR / "best_student_performance_model.pkl"
FINAL_THRESHOLD_PATH = MODELS_DIR / "final_threshold_info.pkl"

print("Final model path:", FINAL_MODEL_PATH)
print("Final threshold path:", FINAL_THRESHOLD_PATH)

Final model path: ..\outputs\models\best_student_performance_model.pkl
Final threshold path: ..\outputs\models\final_threshold_info.pkl


In [3]:
# Cell 4 - Load Final Model and Threshold

final_model = joblib.load(FINAL_MODEL_PATH)
threshold_info = joblib.load(FINAL_THRESHOLD_PATH)

best_threshold = threshold_info["threshold"]

print("Final model loaded successfully.")
print("Best threshold:", best_threshold)
print("Prediction rule:", threshold_info["rule"])

Final model loaded successfully.
Best threshold: 0.8
Prediction rule: If P(Pass) >= threshold, predict Pass; otherwise predict Fail.


In [4]:
# Cell 5 - Load Dataset for Column Reference

df = pd.read_csv(DATA_PATH)

drop_columns = [
    "Exam_Score",
    "Performance_Label",
    "Pass_Fail"
]

feature_columns = df.drop(columns=drop_columns).columns.tolist()

print("Number of input features:", len(feature_columns))
print("\nInput features:")
for col in feature_columns:
    print("-", col)

Number of input features: 19

Input features:
- Hours_Studied
- Attendance
- Parental_Involvement
- Access_to_Resources
- Extracurricular_Activities
- Sleep_Hours
- Previous_Scores
- Motivation_Level
- Internet_Access
- Tutoring_Sessions
- Family_Income
- Teacher_Quality
- School_Type
- Peer_Influence
- Physical_Activity
- Learning_Disabilities
- Parental_Education_Level
- Distance_from_Home
- Gender


In [5]:
# Cell 6 - Show Available Values for Categorical Features

categorical_columns = df.drop(columns=drop_columns).select_dtypes(include=["object"]).columns.tolist()

print("Categorical feature values:")
print("-" * 60)

for col in categorical_columns:
    print(f"\n{col}:")
    print(df[col].dropna().unique())

Categorical feature values:
------------------------------------------------------------

Parental_Involvement:
['Low' 'Medium' 'High']

Access_to_Resources:
['High' 'Medium' 'Low']

Extracurricular_Activities:
['No' 'Yes']

Motivation_Level:
['Low' 'Medium' 'High']

Internet_Access:
['Yes' 'No']

Family_Income:
['Low' 'Medium' 'High']

Teacher_Quality:
['Medium' 'High' 'Low']

School_Type:
['Public' 'Private']

Peer_Influence:
['Positive' 'Negative' 'Neutral']

Learning_Disabilities:
['No' 'Yes']

Parental_Education_Level:
['High School' 'College' 'Postgraduate']

Distance_from_Home:
['Near' 'Moderate' 'Far']

Gender:
['Male' 'Female']


In [6]:
# Cell 7 - Prediction Function

def predict_student_performance(student_data, model, threshold):
    student_df = pd.DataFrame([student_data])

    student_df = student_df[feature_columns]

    pass_probability = float(model.predict_proba(student_df)[:, 1][0])
    fail_probability = float(1 - pass_probability)

    prediction_numeric = 1 if pass_probability >= threshold else 0
    prediction_label = "Pass" if prediction_numeric == 1 else "Fail"

    if fail_probability >= 0.70:
        risk_level = "High Risk"
        recommendation = (
            "Immediate academic intervention is recommended. "
            "The student may need tutoring, attendance monitoring, and advisor support."
        )
    elif fail_probability >= 0.40:
        risk_level = "Medium Risk"
        recommendation = (
            "The student should be monitored. "
            "Extra support such as study planning and tutoring sessions may be helpful."
        )
    else:
        risk_level = "Low Risk"
        recommendation = (
            "The student is currently predicted to perform well. "
            "Continue monitoring progress and encourage good study habits."
        )

    result = {
        "Prediction": prediction_label,
        "Pass_Probability": float(round(pass_probability, 4)),
        "Fail_Probability": float(round(fail_probability, 4)),
        "Risk_Level": risk_level,
        "Recommendation": recommendation
    }

    return result

In [8]:
# Cell 8 - Sample Student 1: Strong Student

strong_student = {
    "Hours_Studied": 28,
    "Attendance": 95,
    "Parental_Involvement": "High",
    "Access_to_Resources": "High",
    "Extracurricular_Activities": "Yes",
    "Sleep_Hours": 8,
    "Previous_Scores": 88,
    "Motivation_Level": "High",
    "Internet_Access": "Yes",
    "Tutoring_Sessions": 2,
    "Family_Income": "High",
    "Teacher_Quality": "High",
    "School_Type": "Private",
    "Peer_Influence": "Positive",
    "Physical_Activity": 3,
    "Learning_Disabilities": "No",
    "Parental_Education_Level": "Postgraduate",
    "Distance_from_Home": "Near",
    "Gender": "Male"
}

strong_result = predict_student_performance(
    student_data=strong_student,
    model=final_model,
    threshold=best_threshold
)

strong_result

{'Prediction': 'Pass',
 'Pass_Probability': 1.0,
 'Fail_Probability': 0.0,
 'Risk_Level': 'Low Risk',
 'Recommendation': 'The student is currently predicted to perform well. Continue monitoring progress and encourage good study habits.'}

In [9]:
# Cell 9 - Sample Student 2: At-Risk Student

at_risk_student = {
    "Hours_Studied": 4,
    "Attendance": 62,
    "Parental_Involvement": "Low",
    "Access_to_Resources": "Low",
    "Extracurricular_Activities": "No",
    "Sleep_Hours": 5,
    "Previous_Scores": 52,
    "Motivation_Level": "Low",
    "Internet_Access": "No",
    "Tutoring_Sessions": 0,
    "Family_Income": "Low",
    "Teacher_Quality": "Low",
    "School_Type": "Public",
    "Peer_Influence": "Negative",
    "Physical_Activity": 1,
    "Learning_Disabilities": "Yes",
    "Parental_Education_Level": "High School",
    "Distance_from_Home": "Far",
    "Gender": "Female"
}

at_risk_result = predict_student_performance(
    student_data=at_risk_student,
    model=final_model,
    threshold=best_threshold
)

at_risk_result

{'Prediction': 'Fail',
 'Pass_Probability': 0.0,
 'Fail_Probability': 1.0,
 'Risk_Level': 'High Risk',
 'Recommendation': 'Immediate academic intervention is recommended. The student may need tutoring, attendance monitoring, and advisor support.'}

In [10]:
# Cell 10 - Sample Student 3: Medium Risk Student

medium_risk_student = {
    "Hours_Studied": 14,
    "Attendance": 75,
    "Parental_Involvement": "Medium",
    "Access_to_Resources": "Medium",
    "Extracurricular_Activities": "Yes",
    "Sleep_Hours": 6,
    "Previous_Scores": 68,
    "Motivation_Level": "Medium",
    "Internet_Access": "Yes",
    "Tutoring_Sessions": 1,
    "Family_Income": "Medium",
    "Teacher_Quality": "Medium",
    "School_Type": "Public",
    "Peer_Influence": "Neutral",
    "Physical_Activity": 2,
    "Learning_Disabilities": "No",
    "Parental_Education_Level": "College",
    "Distance_from_Home": "Moderate",
    "Gender": "Male"
}

medium_result = predict_student_performance(
    student_data=medium_risk_student,
    model=final_model,
    threshold=best_threshold
)

medium_result

{'Prediction': 'Pass',
 'Pass_Probability': 1.0,
 'Fail_Probability': 0.0,
 'Risk_Level': 'Low Risk',
 'Recommendation': 'The student is currently predicted to perform well. Continue monitoring progress and encourage good study habits.'}

In [11]:
# Cell 11 - Combine Demo Results

demo_results = pd.DataFrame([
    {
        "Student Case": "Strong Student",
        **strong_result
    },
    {
        "Student Case": "At-Risk Student",
        **at_risk_result
    },
    {
        "Student Case": "Medium Risk Student",
        **medium_result
    }
])

demo_results

,Student Case,Prediction,Pass_Probability,Fail_Probability,Risk_Level,Recommendation
0,Strong Student,Pass,1.0,0.0,Low Risk,The student is currently predicted to perform ...
1,At-Risk Student,Fail,0.0,1.0,High Risk,Immediate academic intervention is recommended...
2,Medium Risk Student,Pass,1.0,0.0,Low Risk,The student is currently predicted to perform ...


In [12]:
# Cell 12 - Create Cleaner Display Table

demo_display_df = demo_results.copy()

demo_display_df["Pass_Probability"] = demo_display_df["Pass_Probability"].apply(lambda x: f"{x:.2%}")
demo_display_df["Fail_Probability"] = demo_display_df["Fail_Probability"].apply(lambda x: f"{x:.2%}")

demo_display_df

,Student Case,Prediction,Pass_Probability,Fail_Probability,Risk_Level,Recommendation
0,Strong Student,Pass,100.00%,0.00%,Low Risk,The student is currently predicted to perform ...
1,At-Risk Student,Fail,0.00%,100.00%,High Risk,Immediate academic intervention is recommended...
2,Medium Risk Student,Pass,100.00%,0.00%,Low Risk,The student is currently predicted to perform ...


In [13]:
# Cell 13 - Save Demo Results

demo_results_path = TABLES_DIR / "prediction_demo_results.csv"
demo_display_path = TABLES_DIR / "prediction_demo_display_results.csv"

demo_results.to_csv(demo_results_path, index=False)
demo_display_df.to_csv(demo_display_path, index=False)

print("Prediction demo results saved successfully.")
print("Saved to:", demo_results_path)

print("Prediction demo display results saved successfully.")
print("Saved to:", demo_display_path)

Prediction demo results saved successfully.
Saved to: ..\outputs\tables\prediction_demo_results.csv
Prediction demo display results saved successfully.
Saved to: ..\outputs\tables\prediction_demo_display_results.csv


In [14]:
# Cell 14 - Manual Prediction Function

def run_student_prediction(
    Hours_Studied,
    Attendance,
    Parental_Involvement,
    Access_to_Resources,
    Extracurricular_Activities,
    Sleep_Hours,
    Previous_Scores,
    Motivation_Level,
    Internet_Access,
    Tutoring_Sessions,
    Family_Income,
    Teacher_Quality,
    School_Type,
    Peer_Influence,
    Physical_Activity,
    Learning_Disabilities,
    Parental_Education_Level,
    Distance_from_Home,
    Gender
):
    student_data = {
        "Hours_Studied": Hours_Studied,
        "Attendance": Attendance,
        "Parental_Involvement": Parental_Involvement,
        "Access_to_Resources": Access_to_Resources,
        "Extracurricular_Activities": Extracurricular_Activities,
        "Sleep_Hours": Sleep_Hours,
        "Previous_Scores": Previous_Scores,
        "Motivation_Level": Motivation_Level,
        "Internet_Access": Internet_Access,
        "Tutoring_Sessions": Tutoring_Sessions,
        "Family_Income": Family_Income,
        "Teacher_Quality": Teacher_Quality,
        "School_Type": School_Type,
        "Peer_Influence": Peer_Influence,
        "Physical_Activity": Physical_Activity,
        "Learning_Disabilities": Learning_Disabilities,
        "Parental_Education_Level": Parental_Education_Level,
        "Distance_from_Home": Distance_from_Home,
        "Gender": Gender
    }

    result = predict_student_performance(
        student_data=student_data,
        model=final_model,
        threshold=best_threshold
    )

    print("Student Performance Prediction")
    print("-" * 60)
    print(f"Prediction: {result['Prediction']}")
    print(f"Pass Probability: {result['Pass_Probability']:.2%}")
    print(f"Fail Probability: {result['Fail_Probability']:.2%}")
    print(f"Risk Level: {result['Risk_Level']}")
    print("\nRecommendation:")
    print(result["Recommendation"])

    return result

In [15]:
# Cell 15 - Manual Prediction Example

manual_prediction = run_student_prediction(
    Hours_Studied=10,
    Attendance=70,
    Parental_Involvement="Low",
    Access_to_Resources="Medium",
    Extracurricular_Activities="No",
    Sleep_Hours=6,
    Previous_Scores=60,
    Motivation_Level="Low",
    Internet_Access="Yes",
    Tutoring_Sessions=0,
    Family_Income="Low",
    Teacher_Quality="Medium",
    School_Type="Public",
    Peer_Influence="Neutral",
    Physical_Activity=2,
    Learning_Disabilities="No",
    Parental_Education_Level="High School",
    Distance_from_Home="Moderate",
    Gender="Male"
)

Student Performance Prediction
------------------------------------------------------------
Prediction: Fail
Pass Probability: 0.00%
Fail Probability: 100.00%
Risk Level: High Risk

Recommendation:
Immediate academic intervention is recommended. The student may need tutoring, attendance monitoring, and advisor support.


In [18]:
# Cell 16 - Save Demo Summary Text

demo_summary = f"""
Prediction Demo Summary

The final model was applied to three example student cases:

1. Strong Student
2. At-Risk Student
3. Medium Risk Student

For each case, the system produced:
- Pass/Fail prediction
- Pass probability
- Fail probability
- Risk level
- Recommendation

Final model:
{final_model.named_steps["model"].__class__.__name__}

Decision threshold:
{best_threshold}

This demo shows that the project can be used as a practical early-warning system for identifying students who may need academic support.
"""

demo_summary_path = TABLES_DIR / "prediction_demo_summary.txt"

with open(demo_summary_path, "w", encoding="utf-8") as f:
    f.write(demo_summary)

print("Prediction demo summary saved successfully.")
print("Saved to:", demo_summary_path)

Prediction demo summary saved successfully.
Saved to: ..\outputs\tables\prediction_demo_summary.txt


In [19]:
# Cell 17 - Final Output Check

print("06 Prediction Demo Output Files")
print("-" * 60)

important_files = [
    TABLES_DIR / "prediction_demo_results.csv",
    TABLES_DIR / "prediction_demo_display_results.csv",
    TABLES_DIR / "prediction_demo_summary.txt"
]

for file in important_files:
    print(file.name, "->", "Found" if file.exists() else "Not found")

06 Prediction Demo Output Files
------------------------------------------------------------
prediction_demo_results.csv -> Found
prediction_demo_display_results.csv -> Found
prediction_demo_summary.txt -> Found


# Cell 18 - Markdown
"""
## Final Project Notebook Summary

The complete machine learning project is now implemented from A to Z.

Completed notebooks:

1. 01_data_cleaning.ipynb
   - Cleaned the raw dataset
   - Handled missing values
   - Corrected invalid scores
   - Created Pass/Fail target

2. 02_eda_feature_engineering.ipynb
   - Analyzed dataset statistics
   - Created visualizations
   - Studied class imbalance and correlations

3. 03_baseline_modeling.ipynb
   - Trained baseline models
   - Compared Logistic Regression, Decision Tree, Random Forest, SVM, and Gradient Boosting

4. 04_imbalance_and_tuning.ipynb
   - Handled class imbalance
   - Applied threshold tuning
   - Applied hyperparameter tuning
   - Selected the best candidate model

5. 05_final_model_explainability.ipynb
   - Evaluated the final model
   - Created final confusion matrix
   - Explained feature influence using permutation importance and coefficients

6. 06_prediction_demo.ipynb
   - Demonstrated practical use of the final model
   - Predicted Pass/Fail for sample students
   - Provided risk levels and recommendations

The project is now ready for the final report and presentation.
"""